In [1]:
import numpy as np
import shapely
import warnings
import zarr
from zarr.dtype import VariableLengthBytes, VariableLengthUTF8
from zarr.errors import UnstableSpecificationWarning

warnings.filterwarnings("ignore", category=UnstableSpecificationWarning)

### Create Zarr store using meta group

In [ ]:
root = zarr.open_group("my_store.zarr", mode="w", zarr_format=3)
meta = root.create_group("meta")

# Timestamps
meta.create_array(
    "date",
    data=np.array(["2023-01-01", "2023-01-02", "2023-01-03"], dtype="datetime64[ms]"),
)

# String metadata
collection = meta.create_array(
    "collection",
    shape=(3,),
    dtype=VariableLengthUTF8(),
)
collection[:] = ["sentinel-2", "sentinel-2", "landsat-8"]

# Bounding boxes stored as WKB
bbox = meta.create_array(
    "bbox",
    shape=(3,),
    dtype=VariableLengthBytes(),
)
bbox[:] = shapely.to_wkb([
    shapely.box(-10.0, -10.0, 10.0, 10.0),
    shapely.box(-20.0, -20.0, 20.0, 20.0),
    shapely.box( 30.0,  30.0, 50.0, 50.0),
])

### Register Zarr store with custom TableProvider

In [9]:
from datafusion import SessionContext
from obstore.store import LocalStore
from zarr_datafusion_search import ZarrTable

store = LocalStore("my_store.zarr")
zarr_table = await ZarrTable.from_obstore(store, "/meta")

ctx = SessionContext()
ctx.register_table("my_data", zarr_table)

/var/folders/mp/33cxt8xj36bbxj0jqdwbfwyc0000gn/T/ipykernel_85861/1827052559.py:6: RuntimeWarning: Successfully reconstructed a store defined in another Python module. Connection pooling will not be shared across store instances.
  zarr_table = await ZarrTable.from_obstore(store, "/meta")


### Query metadata

In [7]:
df = ctx.sql("SELECT * FROM my_data")
print(df.schema())
df.show()

# Filter by date
df = ctx.sql("""
    SELECT date, collection
    FROM my_data
    WHERE date >= '2023-01-02'
""")
df.show()

bbox: binary_view not null
  -- field metadata --
  ARROW:extension:name: 'geoarrow.wkb'
  ARROW:extension:metadata: '{"crs":"EPSG:4326","crs_type":"authority_cod' + 3
collection: string_view not null
date: timestamp[ms] not null
DataFrame()
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------+
| bbox                                                                                                                                                                                       | collection | date                |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------+
| 01030000000100000005000000000000000000244000000000000024c000000000000024400000

### Query spatial metadata

In [8]:
from geodatafusion import register_all

register_all(ctx)

df = ctx.sql("""
    SELECT date, collection
    FROM my_data
    WHERE ST_Intersects(
        bbox,
        ST_GeomFromText('POLYGON((-15 -15, -15 15, 15 15, 15 -15, -15 -15))')
    )
""")
df.show()

DataFrame()
+---------------------+------------+
| date                | collection |
+---------------------+------------+
| 2023-01-01T00:00:00 | sentinel-2 |
| 2023-01-02T00:00:00 | sentinel-2 |
+---------------------+------------+
